# Module 4: Deploy to Production

You've trained the model, optimized it, and exported it. Now let's ship it.

In this module, you'll:

1. Build a **FastAPI inference API** with proper error handling
2. **Containerize** it with Docker (production-grade Dockerfile)
3. **Deploy** to Google Cloud Run
4. Understand what to **monitor** in production

---

## 4.1 The Inference API

Let's walk through the FastAPI server at `serve/app.py`.

Key production patterns:
- **Model loaded once at startup** (not per-request)
- **Health endpoint** (`/health`) for load balancers
- **Input validation** with Pydantic
- **Structured error responses**
- **Request timing** middleware

In [ ]:
import sys
sys.path.insert(0, '..')

# Let's look at the serving code
from pathlib import Path

app_code = Path('../serve/app.py').read_text()
print(app_code)

### Anatomy of the API

```
POST /generate
{
    "prompt": "The meaning of life is",
    "max_tokens": 50,
    "temperature": 0.8
}
→
{
    "text": "The meaning of life is ...",
    "tokens_generated": 50,
    "latency_ms": 123.4
}
```

```
GET /health
→
{
    "status": "healthy",
    "model_loaded": true,
    "device": "cpu"
}
```

## 4.2 Run Locally

First, test the server locally before containerizing.

In [ ]:
# The server expects a checkpoint and tokenizer in the serve/ directory.
# Let's copy them there.
import shutil

serve_dir = Path('../serve')
exports_dir = Path('../exports')
exports_dir.mkdir(exist_ok=True)

# Copy tokenizer
tokenizer_src = Path('../tokenizer.json')
if tokenizer_src.exists():
    shutil.copy(tokenizer_src, serve_dir / 'tokenizer.json')
    print(f"Copied tokenizer to {serve_dir / 'tokenizer.json'}")

# Copy latest checkpoint
from src.utils import CheckpointManager
ckpt_manager = CheckpointManager('../checkpoints')
latest = ckpt_manager.latest()
if latest:
    shutil.copy(latest, serve_dir / 'model_checkpoint.pt')
    print(f"Copied checkpoint to {serve_dir / 'model_checkpoint.pt'}")
else:
    print("No checkpoint found — save one first by running Module 1.")

### Run the server in a terminal:

```bash
cd serve
uvicorn app:app --host 0.0.0.0 --port 8000 --reload
```

Then test with curl:

```bash
# Health check
curl http://localhost:8000/health

# Generate text
curl -X POST http://localhost:8000/generate \
  -H "Content-Type: application/json" \
  -d '{"prompt": "The future of AI", "max_tokens": 50}'
```

In [ ]:
# Or test programmatically (run this after starting the server)
import requests

BASE_URL = "http://localhost:8000"

try:
    # Health check
    resp = requests.get(f"{BASE_URL}/health", timeout=5)
    print("Health:", resp.json())
    
    # Generate
    resp = requests.post(
        f"{BASE_URL}/generate",
        json={"prompt": "The future of AI", "max_tokens": 50, "temperature": 0.8},
        timeout=30,
    )
    result = resp.json()
    print(f"\nGenerated text: {result['text'][:200]}")
    print(f"Latency: {result['latency_ms']:.1f} ms")
    
except requests.ConnectionError:
    print("Server not running. Start it with:")
    print("  cd ../serve && uvicorn app:app --host 0.0.0.0 --port 8000")

## 4.3 Docker: Production Container

Let's look at the Dockerfile. Key production patterns:

- **Multi-stage build**: Separate build and runtime stages
- **Non-root user**: Security best practice
- **Health check**: Docker-level health monitoring
- **Minimal image**: Only include what's needed at runtime

In [ ]:
# View the Dockerfile
dockerfile = Path('../serve/Dockerfile').read_text()
print(dockerfile)

### Build and run:

```bash
cd serve

# Build the image
docker build -t pytorch-workshop-api .

# Run the container
docker run -p 8000:8000 pytorch-workshop-api

# Test it
curl http://localhost:8000/health
curl -X POST http://localhost:8000/generate \
  -H "Content-Type: application/json" \
  -d '{"prompt": "Hello world", "max_tokens": 30}'
```

## 4.4 Deploy to Google Cloud Run

Cloud Run is a managed container platform — you give it a Docker image and it handles scaling, HTTPS, and load balancing.

### Prerequisites:
- `gcloud` CLI installed and authenticated
- A GCP project with billing enabled
- Cloud Run API enabled

### Deploy:

```bash
# Set your project
export GCP_PROJECT=your-project-id
export REGION=us-central1
export SERVICE_NAME=pytorch-workshop-api

# Build and push to Container Registry
gcloud builds submit --tag gcr.io/$GCP_PROJECT/$SERVICE_NAME serve/

# Deploy to Cloud Run
gcloud run deploy $SERVICE_NAME \
  --image gcr.io/$GCP_PROJECT/$SERVICE_NAME \
  --platform managed \
  --region $REGION \
  --port 8000 \
  --memory 2Gi \
  --cpu 2 \
  --timeout 60 \
  --concurrency 10 \
  --min-instances 0 \
  --max-instances 5 \
  --allow-unauthenticated
```

### Or use the deploy script:

```bash
cd serve
chmod +x deploy.sh
./deploy.sh your-project-id
```

In [ ]:
# View the deploy script
deploy_script = Path('../serve/deploy.sh').read_text()
print(deploy_script)

## 4.5 What to Monitor in Production

Once deployed, you need to know when things go wrong. Key metrics:

| Metric | What it tells you | Alert threshold |
|--------|------------------|----------------|
| **Request latency (P99)** | User experience | >500ms for sync APIs |
| **Error rate** | Model/service health | >1% |
| **Memory usage** | OOM risk | >80% of limit |
| **Cold start time** | Time to first request | >10s |
| **Model staleness** | When was the model last updated | App-specific |

### Cloud Run gives you:
- Request count, latency, error rate (built-in)
- Container CPU/memory usage (built-in)
- Custom metrics via Cloud Logging (add to your app)

### Application-level logging (already in our API):
- Request ID for tracing
- Inference latency per request
- Input/output sizes

## Summary: What We Built Today

```
┌─────────────────────────────────────────────────────────┐
│                  Workshop Pipeline                      │
│                                                         │
│  ┌─────────┐   ┌──────────┐   ┌──────────┐   ┌──────┐ │
│  │ Module 1 │──▶│ Module 2 │──▶│ Module 3 │──▶│Mod. 4│ │
│  │ Train   │   │ Optimize │   │ Export   │   │Deploy│ │
│  └─────────┘   └──────────┘   └──────────┘   └──────┘ │
│                                                         │
│  Transformer     AMP           TorchScript    FastAPI   │
│  Training loop   Profiling     ONNX          Docker    │
│  Checkpoints     Stability     Quantization  Cloud Run │
│  Eval + logging  DataLoader    Benchmarking  Monitoring│
└─────────────────────────────────────────────────────────┘
```

You now have a complete, production-grade ML pipeline — from a raw model to a deployed, monitored inference service.

### Next steps for your own projects:
1. **Swap the model**: Replace `TransformerLM` with your model
2. **Scale up**: Add GPU Cloud Run instances for large models
3. **Add CI/CD**: GitHub Actions for automatic deployment
4. **A/B testing**: Deploy multiple model versions side by side
5. **Monitoring**: Add Prometheus metrics or Cloud Monitoring